## 아부다비 그랑프리 fastF1 추출

In [4]:
import logging
import pandas as pd
import fastf1
from meteostat import Point, daily, stations, config
from datetime import timedelta
from geopy.geocoders import Nominatim

logging.getLogger("fastf1").setLevel(logging.WARNING)
config.block_large_requests = False

YEAR = 2022
RACE = "Abu Dhabi"

session = fastf1.get_session(YEAR, RACE, 'R')
session.load(laps=True, telemetry=False, weather=False, messages=False)

laps = session.laps.copy()
laps["TrackStatus"] = laps["TrackStatus"].astype(str)

laps["SafetyCar"] = laps["TrackStatus"].str.contains("4", na=False).astype(int)
laps["VSC"] = laps["TrackStatus"].str.contains("6", na=False).astype(int)
laps["YellowFlag"] = laps["TrackStatus"].str.contains("2", na=False).astype(int)
laps["RedFlag"] = laps["TrackStatus"].str.contains("7", na=False).astype(int)
laps["RecentSC"] = laps["SafetyCar"].rolling(3, min_periods=1).max().astype(int)

event = session.event
race_date = pd.to_datetime(event["EventDate"])

geolocator = Nominatim(user_agent="f1_weather")
geo = geolocator.geocode(f"{event['Location']}, {event['Country']}")

temp = rainfall = windspeed = None

if geo:
    point = Point(geo.latitude, geo.longitude)
    nearby = stations.nearby(point, radius=200000)

    if not nearby.empty:
        weather = daily(nearby, race_date - timedelta(days=1), race_date + timedelta(days=1)).fetch()

        if weather is not None and not weather.empty:
            if isinstance(weather.index, pd.MultiIndex):
                weather = weather.reset_index(level=0, drop=True)

            weather = weather.loc[weather.index.date == race_date.date()]

            if not weather.empty:
                weather = weather.iloc[0]
                tavg = weather.get("tavg")
                tmin = weather.get("tmin")
                tmax = weather.get("tmax")
                if pd.isna(tavg) and not pd.isna(tmin) and not pd.isna(tmax):
                    tavg = (tmin + tmax) / 2
                temp = tavg
                rainfall = weather.get("prcp")
                windspeed = weather.get("wspd")

context_df = laps[["Driver", "LapNumber", "TrackStatus"]].copy()
context_df["SafetyCar"] = laps["SafetyCar"]
context_df["VSC"] = laps["VSC"]
context_df["YellowFlag"] = laps["YellowFlag"]
context_df["RedFlag"] = laps["RedFlag"]
context_df["RecentSC"] = laps["RecentSC"]
context_df["AirTemp"] = temp
context_df["Rainfall"] = rainfall
context_df["WindSpeed"] = windspeed
context_df["Race"] = RACE
context_df["Year"] = YEAR

print(context_df.head())
context_df.to_csv("abu_dhabi_2022_context.csv", index=False)


  Driver  LapNumber TrackStatus  SafetyCar  VSC  YellowFlag  RedFlag  \
0    VER        1.0           1          0    0           0        0   
1    VER        2.0           1          0    0           0        0   
2    VER        3.0           1          0    0           0        0   
3    VER        4.0           1          0    0           0        0   
4    VER        5.0           1          0    0           0        0   

   RecentSC  AirTemp  Rainfall  WindSpeed       Race  Year  
0         0     27.5       0.0        9.5  Abu Dhabi  2022  
1         0     27.5       0.0        9.5  Abu Dhabi  2022  
2         0     27.5       0.0        9.5  Abu Dhabi  2022  
3         0     27.5       0.0        9.5  Abu Dhabi  2022  
4         0     27.5       0.0        9.5  Abu Dhabi  2022  
